<a href="https://colab.research.google.com/github/surajjeoor/masai_assignments/blob/main/Chatbot_for_shop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install transformers
!pip install datasets
!pip install trl
!pip install peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.2/948.2 kB 9.7 MB/s  0:00:00


In [4]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import Dataset
from trl import SFTTrainer, SFTConfig, DPOTrainer, DPOConfig
from peft import LoraConfig

print(torch.cuda.is_available())

True


In [6]:
MODEL_NAME="Qwen/Qwen2.5-0.5B-Instruct"
SYSTEM_PROMPT="You are the Shoppies customer support assistant."
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model=AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto", trust_remote_code=True)
print(p.numel for p in model.parameters())

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

<generator object <genexpr> at 0x78904c98e8c0>


In [7]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Can I refund a used item?"},
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(text)


<|im_start|>system
You are the Shoppies customer support assistant.<|im_end|>
<|im_start|>user
Can I refund a used item?<|im_end|>
<|im_start|>assistant



In [8]:
def make_prompt(question):
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question}]

def ask(model, question, max_new_tokens=60):
    text = tokenizer.apply_chat_template(make_prompt(question), tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    model.eval()
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


In [10]:
# Cell: sft-dataset
# ── ShopEase SFT dataset ──
# Each item has a 'text' field containing the full formatted training example.
# The model learns to predict the Response given the Instruction.

SFT_EXAMPLES = [
    # Returns
    ("What is Shoppies's return policy?",
     "Shoppies accepts returns within 30 days of delivery. Items must be unused, in original packaging, and accompanied by the order receipt."),

    ("Can I return a used item?",
     "We only accept returns for items in original, unused condition. Used items are not eligible for return or refund."),

    ("How do I start a return?",
     "Go to My Orders in your account, select the item, and click 'Request Return'. Our team will send a prepaid return label within 24 hours."),

    ("When will I get my refund after returning?",
     "Refunds are processed within 5–7 business days after we receive the returned item. The amount is credited to your original payment method."),

    ("Can I exchange instead of returning?",
     "Yes. You can request an exchange for a different size or colour during the return process. Exchanges are processed at no extra charge."),

    # Shipping
    ("How long does standard shipping take?",
     "Standard shipping takes 3–5 business days. Express shipping (1–2 days) is available at checkout for an additional fee."),

    ("How do I track my order?",
     "Once your order is shipped, you will receive a tracking number by email. You can also track your order under My Orders in your account."),

    ("Do you offer free shipping?",
     "Orders above ₹999 qualify for free standard shipping. Orders below this amount have a flat shipping fee of ₹49."),

    ("My order says delivered but I haven't received it.",
     "Please check with your neighbours and building reception first. If the package is still missing after 24 hours, contact us and we will investigate immediately."),

    ("Can I change my delivery address after placing an order?",
     "Address changes are possible only if the order has not yet been dispatched. Contact support immediately and we will try to update it for you."),

    # Account
    ("How do I reset my password?",
     "On the login page, click 'Forgot Password', enter your registered email, and follow the link sent to your inbox to reset your password."),

    ("How do I cancel my order?",
     "Orders can be cancelled within 1 hour of placing them. Go to My Orders, select the order, and click 'Cancel'. After 1 hour, cancellation is not possible."),

    ("I was charged twice for my order.",
     "We apologise for this error. Please share your order ID and we will verify the duplicate charge and process a full refund within 2 business days."),

    ("How do I apply a coupon code?",
     "Enter your coupon code in the 'Apply Coupon' field on the checkout page before placing your order. The discount will be applied automatically."),

    ("What payment methods do you accept?",
     "We accept UPI, credit and debit cards (Visa, Mastercard, RuPay), net banking, and ShopEase wallet. Cash on delivery is available for orders below ₹5000."),
]

INSTRUCTION_TEMPLATE = """### Instruction:\n{question}\n\n### Response:\n{answer}"""

sft_data = [
    {"text": INSTRUCTION_TEMPLATE.format(question=q, answer=a)}
    for q, a in SFT_EXAMPLES
]

sft_dataset = Dataset.from_list(sft_data)

print(f"SFT dataset size: {len(sft_dataset)} examples")
print("\nSample example:")
print("-" * 60)
print(sft_dataset[0]["text"])

SFT dataset size: 15 examples

Sample example:
------------------------------------------------------------
### Instruction:
What is Shoppies's return policy?

### Response:
ShopEase accepts returns within 30 days of delivery. Items must be unused, in original packaging, and accompanied by the order receipt.


In [12]:
peft_config=LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM"
)

In [19]:
# Re-load a clean copy of the base model to prevent stacking PEFT adapters
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto", trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

sft_config = SFTConfig(output_dir="shoppies-sft", num_train_epochs=8,
                       per_device_train_batch_size=4, logging_steps=5,bf16=False, fp16=True)

sft_trainer = SFTTrainer(model=model, train_dataset=sft_dataset,
                         processing_class=tokenizer, peft_config=peft_config,
                         args=sft_config)
sft_trainer.train()

sft_model = sft_trainer.model.merge_and_unload()

# Define test_questions to avoid NameError
test_questions = [
    "Can I return a used item?",
    "How long does standard shipping take?",
    "What payment methods do you accept?"
]

sft_answers = [ask(sft_model, q) for q in test_questions]
for q, a in zip(test_questions, sft_answers):
    print(f"Question: {q}\nAnswer: {a}\n{'-'*50}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Step,Training Loss
5,3.112536
10,2.894387
15,2.765834
20,2.673280
25,2.554317
30,2.462396


Question: Can I return a used item?
Answer: Yes, you can definitely return a used item to us for a full refund or exchange if it's not working properly after your purchase. Please provide us with the product description and any photos of the item so we can better understand what went wrong. We'll then work with you to find a solution that
--------------------------------------------------
Question: How long does standard shipping take?
Answer: Standard shipping typically takes 3-5 business days for most orders, depending on your location and whether you're in a rush or not. However, some orders may be expedited to ensure they arrive faster.

If you have any specific questions about your order or need further assistance, feel free to ask
--------------------------------------------------
Question: What payment methods do you accept?
Answer: We currently accept credit cards, PayPal, and Apple Pay for online purchases. We also accept cash on delivery for in-store purchases.
--------------

In [18]:
# Cell: dpo-dataset
# DPO dataset: 10 preference pairs
# Format required by DPOTrainer: columns = prompt, chosen, rejected

DPO_EXAMPLES = [
    {
        "prompt": "### Instruction:\nWhat is Shoppies's return policy?\n\n### Response:",
        "chosen": "Shoppies accepts returns within 30 days of delivery. Items must be unused, in original packaging, with the order receipt.",
        "rejected": "Returns are possible sometimes. Please check our website for the latest terms and conditions which may have changed."
    },
    {
        "prompt": "### Instruction:\nCan I return a used item?\n\n### Response:",
        "chosen": "We only accept returns for items in original, unused condition. Used items are not eligible for return or refund.",
        "rejected": "It depends on the item and situation. Our team will look into it on a case-by-case basis and get back to you at some point."
    },
    {
        "prompt": "### Instruction:\nHow long does standard shipping take?\n\n### Response:",
        "chosen": "Standard shipping takes 3–5 business days. Express shipping (1–2 days) is available at checkout for an additional fee.",
        "rejected": "Shipping timelines vary greatly depending on your location, the warehouse, current demand, weather, and many other factors that are hard to predict in advance."
    },
    {
        "prompt": "### Instruction:\nHow do I track my order?\n\n### Response:",
        "chosen": "Once your order ships, you will receive a tracking number by email. You can also check My Orders in your account.",
        "rejected": "Tracking is available. You should have received an email. Try checking your spam folder or maybe contact the courier directly."
    },
    {
        "prompt": "### Instruction:\nWhen will I get my refund after returning?\n\n### Response:",
        "chosen": "Refunds are processed within 5–7 business days after we receive the returned item, credited to your original payment method.",
        "rejected": "Refunds can take anywhere from a few days to a few weeks. It really depends on your bank and the payment method you used."
    },
    {
        "prompt": "### Instruction:\nHow do I cancel my order?\n\n### Response:",
        "chosen": "Orders can be cancelled within 1 hour of placing them via My Orders. After 1 hour, cancellation is not possible as the order is already being processed.",
        "rejected": "You can try to cancel. Go to your orders and see if a cancel button is there. If not, there is not much we can do unfortunately."
    },
    {
        "prompt": "### Instruction:\nDo you offer free shipping?\n\n### Response:",
        "chosen": "Orders above ₹999 qualify for free standard shipping. Orders below this amount have a flat shipping fee of ₹49.",
        "rejected": "Sometimes we offer free shipping on certain products or during special promotions. Keep an eye on our app for deals and discount offers."
    },
    {
        "prompt": "### Instruction:\nHow do I apply a coupon code?\n\n### Response:",
        "chosen": "Enter your coupon code in the 'Apply Coupon' field on the checkout page before placing your order. The discount applies automatically.",
        "rejected": "Coupons need to be applied somewhere on the checkout page I think. Look around for a field to enter the code before paying."
    },
    {
        "prompt": "### Instruction:\nI was charged twice for my order.\n\n### Response:",
        "chosen": "We apologise for this error. Please share your order ID and we will verify the duplicate charge and process a full refund within 2 business days.",
        "rejected": "Double charges sometimes happen due to technical issues. Wait a few days and if it doesn't resolve on its own then let us know and we will see."
    },
    {
        "prompt": "### Instruction:\nWhat payment methods do you accept?\n\n### Response:",
        "chosen": "We accept UPI, credit and debit cards (Visa, Mastercard, RuPay), net banking, and ShopEase wallet. Cash on delivery is available for orders below ₹5000.",
        "rejected": "We accept most major payment methods. We are always adding new options so the list may be different from what you have seen before."
    },
]

dpo_dataset = Dataset.from_list(DPO_EXAMPLES)

print(f"DPO dataset size: {len(dpo_dataset)} preference pairs")
print("\nSample pair:")
print("-" * 60)
sample = dpo_dataset[0]
print(f"Prompt  : {sample['prompt'].split(chr(10))[1]}")
print(f"Chosen  : {sample['chosen']}")
print(f"Rejected: {sample['rejected']}")

DPO dataset size: 10 preference pairs

Sample pair:
------------------------------------------------------------
Prompt  : What is ShopEase's return policy?
Chosen  : ShopEase accepts returns within 30 days of delivery. Items must be unused, in original packaging, with the order receipt.
Rejected: Returns are possible sometimes. Please check our website for the latest terms and conditions which may have changed.


In [20]:
dpo_config = DPOConfig(
    output_dir="dpo-shoppies",
    beta=0.1,
    loss_type=["sigmoid", "sft"],
    num_train_epochs=8,
    per_device_train_batch_size=4,
    logging_steps=5,
    bf16=False,
    fp16=True
)
dpo_trainer = DPOTrainer(model=sft_model, args=dpo_config, train_dataset=dpo_dataset,
                         processing_class=tokenizer, peft_config=peft_config)
dpo_trainer.train()




Adding EOS to train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Step,Training Loss
5,3.574378
10,3.420024
15,3.455664
20,3.503284


TrainOutput(global_step=24, training_loss=3.483346621195475, metrics={'train_runtime': 12.5977, 'train_samples_per_second': 6.35, 'train_steps_per_second': 1.905, 'total_flos': 16964204841984.0, 'train_loss': 3.483346621195475, 'entropy': 2.0484677851200104, 'num_tokens': 6656.0, 'logits/chosen': -1.6363032055337987, 'logits/rejected': -0.7662211366309168, 'mean_token_accuracy': 0.4965512081980705, 'rewards/chosen': 0.04301729332655668, 'rewards/rejected': 0.025241375085897744, 'rewards/accuracies': 0.9375, 'rewards/margins': 0.017775917891412973, 'logps/chosen': -80.98868560791016, 'logps/rejected': -93.01628303527832, 'epoch': 8.0})

In [22]:
dpo_shoppies_model = dpo_trainer.model.merge_and_unload()

In [24]:
test_questions = [
    "In how many days I can return the item?",
    "How long does standard shipping take?",
    "What payment methods do you accept?"
]

sft_answers = [ask(dpo_shoppies_model, q) for q in test_questions]
for q, a in zip(test_questions, sft_answers):
    print(f"Question: {q}\nAnswer: {a}\n{'-'*50}")

Question: In how many days I can return the item?
Answer: I'm sorry, but as an AI language model, I don't have access to your specific order details or shipping information. However, most online retailers allow customers to return items within 30 days of purchase for a refund or exchange.

To find out if you qualify for a return, check the
--------------------------------------------------
Question: How long does standard shipping take?
Answer: Standard shipping typically takes 3-5 business days for most orders, depending on your location and the time of year. However, some items may take longer to ship due to their size or weight.

If you have any questions about your order's status or need assistance with tracking your package, please feel
--------------------------------------------------
Question: What payment methods do you accept?
Answer: We currently accept credit cards, PayPal, and Apple Pay for online purchases. We also accept cash on delivery for some of our products. Please 